# Flipkart Gridlock 2.0 - Traffic Demand Forecasting
**Approach:** Autoregressive Time-Series Forecasting via Heterogeneous LightGBM Ensemble

### Architecture Overview
Predicting 47 time-steps into the future requires a model that understands momentum. Standard regression fails here because it cannot dynamically update its historical context. 
This notebook implements a **Recursive Forecasting Loop**:
1. Predict the next 15 minutes.
2. Append that prediction to the historical context.
3. Dynamically recalculate all rolling averages and lag features.
4. Predict the next 15 minutes based on the updated context.

To prevent recursive error compounding, we utilize a **Heterogeneous Super-Ensemble**, training multiple variations of LightGBM and averaging their step-by-step predictions to ensure extreme stability across the 24-hour cycle.

In [ ]:
# Install pygeohash if not present in the environment
%pip install pygeohash pandas numpy lightgbm scikit-learn -q

import os
import random
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
import pygeohash as pgh
from sklearn.metrics import r2_score

warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
TRAIN_PATH = 'train.csv'
TEST_PATH = 'test.csv'
SUBMISSION_PATH = 'submission_final.csv'

# Advanced Memory Features
LAG_FEATURES = [1, 2, 4, 8, 12, 24, 96]  # Short-term momentum + Exact 24-hour anchor
ROLLING_WINDOWS = [4, 8, 24]             # Smoothing volatility

DYNAMIC_FEATURE_COLUMNS = [
    *[f'demand_lag_{lag}' for lag in LAG_FEATURES],
    *[f'demand_roll_mean_{window}' for window in ROLLING_WINDOWS],
    *[f'demand_roll_std_{window}' for window in ROLLING_WINDOWS],
]

In [ ]:
print("1. Loading Data and Extracting Spatial-Temporal Features...")

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
train_raw['is_test'] = False
test_raw['is_test'] = True
test_raw['demand'] = np.nan

df = pd.concat([train_raw, test_raw], ignore_index=True, sort=False)

# 1. Cyclical Time Encoding
def parse_time(series):
    parts = series.astype(str).str.split(':', n=1, expand=True)
    return parts[0].astype(int) * 60 + parts[1].astype(int)

df['time_mins'] = parse_time(df['timestamp']).astype(int)
df['time_sin'] = np.sin(2 * np.pi * df['time_mins'] / 1440.0)
df['time_cos'] = np.cos(2 * np.pi * df['time_mins'] / 1440.0)

# 2. Spatial Decoding (Geohash to Lat/Lon)
def decode_geo(val):
    if pd.isna(val): return np.nan, np.nan
    try:
        dec = pgh.decode(str(val))
        return float(dec[0]), float(dec[1])
    except: return np.nan, np.nan

geos = df['geohash'].astype('string').dropna().unique()
lookup = {v: decode_geo(v) for v in geos}
coords = df['geohash'].astype('string').map(lookup)
df['latitude'] = [p[0] if isinstance(p, tuple) else np.nan for p in coords]
df['longitude'] = [p[1] if isinstance(p, tuple) else np.nan for p in coords]

# 3. Spatial Grouping & Imputation
df = df.sort_values(['geohash', 'day', 'time_mins']).reset_index(drop=True)
df['Temperature'] = df.groupby('geohash')['Temperature'].transform(lambda s: s.ffill().bfill())
df['Weather'] = df.groupby('geohash')['Weather'].transform(lambda s: s.ffill().bfill())
df['RoadType'] = df['RoadType'].fillna('Unknown')

print(f"Data Base Shape: {df.shape}")

In [ ]:
print("2. Building Dynamic Lags and Rolling Windows...")

grp = df.groupby('geohash', sort=False)

# Create static lags for historical data
for lag in LAG_FEATURES:
    df[f'demand_lag_{lag}'] = grp['demand'].shift(lag)

# Create rolling windows (shifted by 1 to prevent data leakage)
lag_grp = grp['demand'].shift(1).groupby(df['geohash'], sort=False)
for w in ROLLING_WINDOWS:
    df[f'demand_roll_mean_{w}'] = lag_grp.transform(lambda s: s.rolling(w, min_periods=1).mean())
    df[f'demand_roll_std_{w}'] = lag_grp.transform(lambda s: s.rolling(w, min_periods=1).std())

# Fill early NaNs with spatial baseline averages
df[DYNAMIC_FEATURE_COLUMNS] = df[DYNAMIC_FEATURE_COLUMNS].fillna(df.groupby('geohash')['demand'].transform('mean'))

cat_cols = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
for c in cat_cols: df[c] = df[c].astype('category')
cat_levels = {c: df[c].cat.categories for c in cat_cols}

train_df = df[~df['is_test']].copy()
test_df = df[df['is_test']].copy()

f_cols = ['geohash', 'day', 'time_mins', 'time_sin', 'time_cos', 'latitude', 'longitude', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather'] + DYNAMIC_FEATURE_COLUMNS

# Create validation split (Day 48 for training, early Day 49 for validation)
train_mask = (train_df['day'] == 48)
valid_mask = (train_df['day'] == 49) & (train_df['time_mins'] <= 120)

X_train = train_df.loc[train_mask, f_cols].copy()
y_train = train_df.loc[train_mask, 'demand'].astype(float)
X_valid = train_df.loc[valid_mask, f_cols].copy()
y_valid = train_df.loc[valid_mask, 'demand'].astype(float)

print("Feature Engineering Complete.")

In [ ]:
print("3. Initializing Recursive Inference Engine...")

def get_dyn_feats(hist):
    """Calculates lag and rolling features on the fly based on current memory state."""
    feats = {}
    for lag in LAG_FEATURES:
        feats[f'demand_lag_{lag}'] = hist[-lag] if len(hist) >= lag else np.nan
    for w in ROLLING_WINDOWS:
        vals = hist[-w:]
        if vals:
            feats[f'demand_roll_mean_{w}'] = float(np.mean(vals))
            feats[f'demand_roll_std_{w}'] = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        else:
            feats[f'demand_roll_mean_{w}'] = np.nan
            feats[f'demand_roll_std_{w}'] = np.nan
    return feats

def recursive_predict(model, hist_df, tgt_df):
    """
    Steps through time chronologically. Generates a prediction, 
    appends it to the history, and uses it to calculate features for the next step.
    """
    stat_cols = [c for c in f_cols if c not in DYNAMIC_FEATURE_COLUMNS]
    preds, p_idx = [], []
    
    # Initialize spatial memory banks
    hist_map = {str(g): grp.sort_values(['day', 'time_mins'])['demand'].astype(float).tolist()
                for g, grp in hist_df.groupby('geohash', sort=False)}

    for g_val, g_grp in tgt_df.sort_values(['geohash', 'day', 'time_mins']).groupby('geohash', sort=False):
        g_key = str(g_val)
        h_vals = hist_map.get(g_key, [])
        
        for row in g_grp.to_dict('records'):
            f_row = {c: row[c] for c in stat_cols}
            f_row.update(get_dyn_feats(h_vals))  # Dynamically pull autoregressive features
            f_df = pd.DataFrame([f_row], columns=f_cols)
            
            for c in cat_cols: 
                f_df[c] = pd.Categorical(f_df[c], categories=cat_levels[c])
            
            p = float(np.asarray(model.predict(f_df))[0])
            p = max(0.0, p) # Traffic cannot drop below 0
            
            preds.append(p)
            p_idx.append(int(row['Index']))
            
            # CRITICAL: Append prediction to history to predict the next step
            h_vals.append(p)

    return pd.Series(preds, index=p_idx)

def align_predictions(pred_series, index_list):
    mapping = {int(k): float(v) for k, v in pred_series.to_dict().items()}
    preds = np.array([mapping.get(idx, np.nan) for idx in index_list], dtype=float)
    if np.isnan(preds).any():
        preds = np.where(np.isnan(preds), 0.0, preds)
    return preds

In [ ]:
print("4. Training Heterogeneous Ensemble Models...")

N_TRIALS = 20
TOP_K = 5 

def sample_params(rng):
    return {
        'learning_rate': rng.choice([0.01, 0.02, 0.03, 0.05]),
        'num_leaves': rng.choice([63, 127, 255]),
        'min_child_samples': rng.choice([5, 10, 20, 40]),
        'subsample': rng.choice([0.7, 0.8, 0.9, 1.0]),
        'colsample_bytree': rng.choice([0.7, 0.8, 0.9, 1.0]),
        'reg_alpha': rng.choice([0.01, 0.1, 0.5]),
        'reg_lambda': rng.choice([0.01, 0.1, 0.5]),
    }

rng = random.Random(123)
trials = []

for i in range(N_TRIALS):
    params = sample_params(rng)
    
    # Try-Except block ensures the code runs on both local CPUs or Kaggle GPUs safely
    try:
        model = lgb.LGBMRegressor(
            objective='regression', metric='rmse', verbosity=-1, max_bin=255,
            device_type='gpu', n_estimators=1500, random_state=i, **params
        )
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], categorical_feature=cat_cols, callbacks=[lgb.early_stopping(50, verbose=False)])
    except:
        model = lgb.LGBMRegressor(
            objective='regression', metric='rmse', verbosity=-1, max_bin=255,
            device_type='cpu', n_jobs=-1, n_estimators=1500, random_state=i, **params
        )
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], categorical_feature=cat_cols, callbacks=[lgb.early_stopping(50, verbose=False)])

    # Validate against actual recursive performance
    val_hist = train_df.loc[train_df['day'] == 48].copy()
    val_target = train_df.loc[valid_mask].copy()
    
    val_series = recursive_predict(model, val_hist, val_target)
    val_preds = align_predictions(val_series, val_target['Index'].astype(int).tolist())
    val_r2 = r2_score(y_valid, val_preds)

    trials.append({'model': model, 'r2': val_r2})
    print(f"   -> Model {i+1}/{N_TRIALS} | Validation R2 = {val_r2:.6f}")

trials_sorted = sorted(trials, key=lambda x: x['r2'], reverse=True)
top_trials = trials_sorted[:TOP_K]
print(f"\nTop {TOP_K} models selected for final ensemble blending.")

In [ ]:
print("5. Generating Final Test Predictions via Super-Ensemble...")

idx_list = test_df['Index'].astype(int).tolist()
ensemble_test = np.zeros(len(idx_list), dtype=float)

# Average the predictions of the top models to neutralize recursive drift
for idx, t in enumerate(top_trials):
    print(f"   -> Executing recursive forecast for Top Model {idx+1}...")
    s = recursive_predict(t['model'], train_df.copy(), test_df.copy())
    ensemble_test += align_predictions(s, idx_list)

ensemble_test /= len(top_trials)

submission_ensemble = pd.DataFrame({'Index': idx_list, 'demand': ensemble_test})
submission_ensemble.to_csv(SUBMISSION_PATH, index=False)

print(f"\n✅ PIPELINE COMPLETE. Submission saved to {SUBMISSION_PATH}")